# CS231 — Deep Learning: Waste Classification
**Member 2** · Custom CNN, ResNet18, MobileNetV4, Grad-CAM and Classical CV comparison.

⚠️ This notebook contains no precomputed accuracy. Run on real data with the same split as the teammate. Execute from the project root.

## 1. Verify environment and repository

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
from src.base_predictor import CLASS_NAMES
print("Working directory:", Path.cwd())
print("PyTorch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
assert Path("src/base_predictor.py").exists(), "Run notebook with CS231 repository as working directory"
print("Class order:", CLASS_NAMES)

## 2. Audit and exploratory data analysis
Do not use synthetic images in the final experiment. Audit flags exact image copies appearing across train/validation/test.

In [ ]:
from src.member2_deeplearning.audit_data import audit_dataset
audit = audit_dataset("data/standard_split")
display(pd.DataFrame(audit["counts"]).fillna(0))
print("Dataset manifest SHA256:", audit["manifest_sha256"])

In [ ]:
distribution = pd.DataFrame(audit["counts"]).fillna(0).reindex(CLASS_NAMES)
distribution.plot(kind="bar", figsize=(10, 4), title="Image count by class / split")
plt.ylabel("Number of images")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
from src.member2_deeplearning.dataset_aug import WasteDataset, get_transforms
train_ds = WasteDataset("data/standard_split/train", CLASS_NAMES, get_transforms(False))
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for idx, ax in enumerate(axes.ravel()):
    paths = [path for path, c in train_ds.samples if c == idx]
    bgr = cv2.imread(paths[0]); rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    ax.imshow(rgb); ax.set_title(CLASS_NAMES[idx]); ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Train real models
Hyperparameters can be adjusted using validation only. Each run creates a *real* model checkpoint in `weights/`. Start with ResNet18; train the others to compare DL architectures.

In [ ]:
def train_model(name, epochs, freeze_epochs=0, batch_size=32, lr=3e-4):
    args = [sys.executable, "-m", "src.member2_deeplearning.train_cnn",
            "--model", name, "--epochs", str(epochs), "--freeze_epochs", str(freeze_epochs),
            "--batch_size", str(batch_size), "--lr", str(lr), "--workers", "0"]
    subprocess.run(args, check=True)

# UNCOMMENT to train the first Transfer Learning model:
# train_model("resnet18", 25, freeze_epochs=3)

In [ ]:
# Additional experiments — uncomment when ready:
# train_model("light_cnn", 35, freeze_epochs=0)
# train_model("mobilenetv4_conv_small", 30, freeze_epochs=3)

## 4. Inspect training history and model selection (validation only)

In [ ]:
models = ["light_cnn", "resnet18", "mobilenetv4_conv_small"]
records=[]
fig, ax = plt.subplots(figsize=(9, 5))
for name in models:
    hist = Path(f"results/dl/{name}/history.csv")
    summ = Path(f"results/dl/{name}/summary.json")
    if not hist.exists() or not summ.exists():
        continue
    h=pd.read_csv(hist); s=json.loads(summ.read_text())
    ax.plot(h["epoch"], h["val_macro_f1"], label=name)
    records.append({"model":name,"best_epoch":s["best_epoch"],
                    "best_val_macro_f1":s["best_val_macro_f1"]})
ax.set(xlabel="Epoch", ylabel="Validation Macro F1", title="Validation curves")
if records: ax.legend()
plt.show()
display(pd.DataFrame(records))

## 5. Evaluate on held-out test
Run **after model selection has finished**. For a real benchmark the Classical SVM checkpoint must also be trained, and both models must use exactly the same test images.

In [ ]:
chosen_checkpoint = "weights/resnet18_best.pth"  # Replace with your VALIDATION-selected model
# DL-only evaluation (remove comment once ready):
# subprocess.run([sys.executable, "-m", "src.evaluation.benchmark", "--dl_only",
#                 "--dl_weights", chosen_checkpoint], check=True)
# Full comparison (requires trained Classical checkpoint):
# subprocess.run([sys.executable, "-m", "src.evaluation.benchmark",
#                 "--dl_weights", chosen_checkpoint,
#                 "--ml_weights", "weights/classical_svm_pipeline.pkl"], check=True)

In [ ]:
comparison_file = Path("results/benchmark_comparison.csv")
if comparison_file.exists():
    display(pd.read_csv(comparison_file))
else:
    print("No measured test metrics yet. Run the benchmark after training.")

## 6. Grad-CAM explanation
The color overlay comes from gradients of the predicted class, not a synthetic pattern. Do not treat it as proof of causal relevance.

In [ ]:
from src.member2_deeplearning.predictor_dl import DeepLearningPredictor
if Path(chosen_checkpoint).exists():
    predictor = DeepLearningPredictor(chosen_checkpoint)
    path, true_id = train_ds.samples[0]  # Illustration only, NOT test evaluation
    img = cv2.cvtColor(cv2.resize(cv2.imread(path), (224, 224)), cv2.COLOR_BGR2RGB)
    out = predictor.predict(img, with_explanation=True)
    print("Predicted:", out["class_name"], "confidence:", round(out["confidence"],3),
          "inference(ms):", out["latency_ms"], "GradCAM(ms):", out["explanation_latency_ms"])
    fig, axes = plt.subplots(1,2,figsize=(9,4))
    axes[0].imshow(img); axes[0].set_title("Original")
    axes[1].imshow(out["visual_explanation"]); axes[1].set_title("Grad-CAM")
    for ax in axes: ax.axis("off")
    plt.show(); predictor.close()
else:
    print("Train the selected model first; no fake Grad-CAM will be shown.")

## 7. Report prompts
Explain (1) train/val/test protocol and leakage audit, (2) differences between scratch and pretrained models, (3) per-class confusion, (4) latency on equal CPU hardware, (5) when Traditional HOG/LBP/HSV succeeds or fails, (6) Grad-CAM weaknesses and domain shift on Vietnam real-world waste images.